# Transversal analyisis
- Main Objective: To identify the largest possible cohort of patients consistently present across all backbone project files, ensuring that subsequent analyses are performed on a harmonized dataset with complete longitudinal and multimodal information.

## Libaries

In [1]:
import pandas as pd 
import json
import os
import matplotlib.pyplot as plt
import time 
import re
from typing import Optional, Set
import numpy as np
from collections import defaultdict

## Summary of the structure
These tables are the backbone:

- Participant Status + Demographics → define the population.

- MDS-UPDRS + LEDD → clinical progression and medication.

- UPSIT, MoCA, DaTscan, Family History → complementary data.

Biospecimens, genetics, digital sensors, FOUND, Remote/Online data complementary to the research.

#### First Step: create a history record of the data 

In [2]:
IGNORE_NAMES = {'.DS_Store'}
IGNORE_PREFIXES = {'.'}  # Oculta archivos/carpetas "dotfiles" (.*)

def is_ignored(name: str) -> bool:
    if name in IGNORE_NAMES:
        return True
    return any(name.startswith(p) for p in IGNORE_PREFIXES)

def safe_file_info(path, name):
    """Devuelve el diccionario de archivo o un error específico."""
    try:
        st = os.stat(path, follow_symlinks=False)
        return {
            "name": name,
            "type": "file",
            "size_bytes": st.st_size,
            # ISO-8601 en UTC (sufijo Z)
            "modified": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(st.st_mtime))
        }
    except PermissionError:
        return {"name": name, "type": "error", "error": "ACCESS_DENIED"}
    except FileNotFoundError:
        # Enlaces rotos o carreras de E/S
        return {"name": name, "type": "error", "error": "NOT_FOUND"}

def folder_to_dict(path):
    # Construye un árbol limpio y ordenado. Omite ocultos y añade metadatos.
    node = {"name": os.path.basename(path) or path, "type": "folder", "children": []}

    try:
        items = [i for i in os.listdir(path) if not is_ignored(i)]
    except PermissionError:
        node["children"].append({"name": "ACCESS_DENIED", "type": "error"})
        return node
    except FileNotFoundError:
        return {"name": path, "type": "error", "error": "NOT_FOUND"}

    items.sort(key=str.lower)

    for item in items:
        p = os.path.join(path, item)
        try:
            # No seguimos enlaces para evitar bucles
            if os.path.islink(p):
                node["children"].append({"name": item, "type": "symlink"})
            elif os.path.isdir(p):
                node["children"].append(folder_to_dict(p))
            else:
                node["children"].append(safe_file_info(p, item))
        except PermissionError:
            node["children"].append({"name": item, "type": "error", "error": "ACCESS_DENIED"})
        except FileNotFoundError:
            node["children"].append({"name": item, "type": "error", "error": "NOT_FOUND"})

    return node

def folder_to_json(root_path, output_file):
    # Crea la carpeta de salida si hace falta
    outdir = os.path.dirname(output_file)
    if outdir:
        os.makedirs(outdir, exist_ok=True)

    tree = folder_to_dict(root_path)
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(tree, f, indent=2, ensure_ascii=False)


folder_to_json(
    '/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025',
    '/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/DATA_COLLECTION_STRUCTURE/Structure.json')


# Patient Presence Matrix Across CSV Files

## Objective
We need to determine whether each patient (identified by **PATNO**) appears in the different `.csv` files.  
The main file `PATIENT_STATUS.csv` provides the list of patient identifiers belonging to the PPMI Clinical cohort.

---

## Approach

1. **Patient Indexing**  
   - Use **PATNO** as the unique index across all `.csv` files.

2. **Building a Presence Matrix**  
   - Construct a dictionary-like (or tabular) structure where:  
     - **Rows (Index):** Each patient’s PATNO.  
     - **Columns:** Names of the `.csv` files.  
     - **Values:** Boolean flags (`True`/`False`) indicating whether the patient appears in the corresponding file.  

3. **Cohort Filtering**  
   - Restrict the patient list to those found in `PATIENT_STATUS.csv`.  
   - For each patient in this cohort, check if their PATNO is present in the other `.csv` files (representing whether a specific test was applied).

---

## Example (simplified)

| PATNO | DEMOGRAPHICS.csv | BIOMARKERS.csv | MRI.csv | ... |
|-------|------------------|----------------|---------|-----|
| 1001  | True             | False          | True    | ... |
| 1002  | True             | True           | False   | ... |
| 1003  | False            | False          | True    | ... |

---

## Outcome
This **presence matrix** provides a clear overview of which tests were applied to each patient based on their occurrence in the respective `.csv` files.


In [3]:
# ========= CONFIG =========
BASE_PATH = "/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025"
OUTPUT_FOLDER = "/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/DATA_COLLECTION_STRUCTURE"
OUTPUT_MATRIX = "PATNO_presence_matrix.csv"
REFERENCE_FILE = "/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Motor Assesments/Motor___MDS-UPDRS/DATA/MDS-UPDRS_Part_III_12Sep2025.csv"
# ==========================

def is_hidden(name: str) -> bool:
    return name.startswith(".")

def list_csvs(base_path):
    csvs = []
    for root, dirs, files in os.walk(base_path):
        dirs[:] = [d for d in dirs if not is_hidden(d)]  # ignora carpetas ocultas
        for f in files:
            if is_hidden(f):
                continue
            if f.lower().endswith(".csv"):
                csvs.append(os.path.join(root, f))
    csvs.sort()
    return csvs

def find_patno_column(columns):
    # PATNO consistente: exacto (ignorando mayúsculas/espacios)
    for c in columns:
        if c.strip().lower() == "patno":
            return c
    return None

def read_patnos(csv_path):
    # leer solo encabezado para detectar columna
    try:
        header = pd.read_csv(csv_path, nrows=0)
    except Exception as e:
        print(f"[skip] header read failed: {csv_path} | {e}")
        return set()

    col = find_patno_column(header.columns)
    if not col:
        print(f"[skip] no PATNO column: {csv_path}")
        return set()

    # leer solo la columna PATNO; limpiar y devolver únicos
    try:
        s = pd.read_csv(csv_path, usecols=[col], dtype={col: "string"}, low_memory=False)[col]
        s = s.astype("string").str.strip()
        s = s[~s.str.lower().isin({"", "nan", "none", "na"})]
        return set(s.dropna().unique().tolist())
    except Exception as e:
        print(f"[skip] read error: {csv_path} | {e}")
        return set()

def unique_name_factory():
    counts = {}
    def unique_name(base):
        n = counts.get(base, 0) + 1
        counts[base] = n
        return base if n == 1 else f"{base} #{n}"
    return unique_name

def build_presence_matrix(base_path, reference_file):
    # 1) filas = PATNO del archivo de referencia
    ref_patnos = read_patnos(reference_file)
    if not ref_patnos:
        raise RuntimeError("No PATNOs found in reference file.")

    rows = sorted(ref_patnos)
    df = pd.DataFrame(False, index=rows, columns=[])

    # generador de nombres únicos basado en basename
    uniq = unique_name_factory()

    # 2) añadir SIEMPRE la columna del archivo de referencia (basename)
    ref_col = uniq(os.path.basename(reference_file))
    df[ref_col] = False
    df.loc[list(ref_patnos), ref_col] = True  # todos True en la columna de referencia

    # 3) otros CSV (excluir el de referencia para no duplicar)
    ref_norm = os.path.normpath(reference_file)
    all_csvs = list_csvs(base_path)
    other_csvs = [p for p in all_csvs if os.path.normpath(p) != ref_norm]

    # 4) añadir columnas solo si hay solapamiento; nombre = basename del archivo
    for p in other_csvs:
        pats = read_patnos(p)
        if not pats:
            continue
        overlap = pats & ref_patnos
        if not overlap:
            continue
        col_name = uniq(os.path.basename(p))
        df[col_name] = False
        df.loc[list(overlap), col_name] = True

    df.index.name = "PATNO"
    return df

def main():
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    matrix = build_presence_matrix(BASE_PATH, REFERENCE_FILE)
    out_path = os.path.join(OUTPUT_FOLDER, OUTPUT_MATRIX)
    matrix.to_csv(out_path, index=True)
    print(f"✅ Presence matrix saved to: {out_path}")

if __name__ == "__main__":
    main()


[skip] no PATNO column: /Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Biospecimen/Biosample_Inventory/DATA/IUSM_ASSAY_DEV_CATALOG_12Sep2025.csv
[skip] no PATNO column: /Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Biospecimen/Biosample_Inventory/DATA/IUSM_BIOSPECIMEN_CELL_CATALOG_12Sep2025.csv
[skip] no PATNO column: /Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Biospecimen/Biosample_Inventory/DATA/IUSM_CATALOG_12Sep2025.csv
[skip] no PATNO column: /Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Biospecimen/Biospecimen_Analysis_Methods/DOCS/PPMI_Project_151_pqtl_Analysis_Annotations_20210210.csv


/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_64943/3460892101.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = False
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_64943/3460892101.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = False
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_64943/3460892101.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

[skip] no PATNO column: /Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases-Online/PPMI_Online_Codebook_12Sep2025.csv
[skip] no PATNO column: /Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases-Online/PPMI_Online_Dictionary_12Sep2025.csv
[skip] no PATNO column: /Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Code_List_-_Harmonized_12Sep2025.csv
[skip] no PATNO column: /Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Code_List_-__Annotated__12Sep2025.csv
[skip] no PATNO column: /Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Data_Dictionary_-_Harmonized_12Sep2025.csv
[skip] no PATNO column: /Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/12_09_2025/Study_Docs/Data___Databases/DATA/Data_Dictionary_-__Annotated__12Sep202

/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_64943/3460892101.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = False
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_64943/3460892101.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = False
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_64943/3460892101.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

✅ Presence matrix saved to: /Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/DATA_COLLECTION_STRUCTURE/PATNO_presence_matrix.csv


/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_64943/3460892101.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = False
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_64943/3460892101.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col_name] = False
/var/folders/vr/_v9wq92941bflv75jsh8kq800000gn/T/ipykernel_64943/3460892101.py:92: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

In [4]:
patno_df = pd.read_csv('/Users/fserracrespi/Desktop/PD_Project_UofL/DATA_COLLECTION/DATA_COLLECTION_STRUCTURE/PATNO_presence_matrix.csv')
patno_df.set_index('PATNO', inplace=True)
patno_df.head()

,MDS-UPDRS_Part_III_12Sep2025.csv,Whole_Blood_Substudy_12Sep2025.csv,iPSC_Catalog_Metadata_12Sep2025.csv,Current_Biospecimen_Analysis_Results_12Sep2025.csv,Deprecated_Biospecimen_Analysis_Results_12Sep2025.csv,Pilot_Biospecimen_Analysis_Results_12Sep2025.csv,Project_181_Adaptive_Immune_Markers_for_Predicting_Cognitive_Dec_12Sep2025.csv,SAA_Biospecimen_Analysis_Results_12Sep2025.csv,Blood_Chemistry___Hematology_12Sep2025.csv,Clinical_Labs_12Sep2025.csv,...,PPMI_Project_9001_20250624_12Sep2025.csv,iu_genetic_consensus_20250515_12Sep2025.csv,Head_Injuries__Online__12Sep2025.csv,Participant_Status_12Sep2025.csv,Physical_Activity__Online__13Sep2025.csv,Smoking_History__Online__12Sep2025.csv,Age_at_visit_12Sep2025.csv,Demographics_12Sep2025.csv,Socio-Economics_12Sep2025.csv,Subject_Cohort_History_12Sep2025.csv
PATNO,,,,,,,,,,,,,,,,,,,,,
100001,True,False,False,True,False,False,False,True,True,True,...,False,True,True,True,True,True,True,True,True,False
100002,True,False,False,False,False,False,False,True,True,True,...,False,True,False,True,False,False,True,True,True,False
100005,True,False,False,False,False,False,False,True,True,True,...,False,True,False,True,False,False,True,True,True,False
100006,True,False,False,True,False,False,False,True,True,True,...,False,True,True,True,True,True,True,True,True,False
100007,True,False,False,True,False,False,False,True,True,True,...,False,True,False,True,False,False,True,True,True,False
